# 第 6 周 · 周三：活动效果评估 SQL——区分自然增长与真实增量

> 难度：⭐⭐⭐⭐ | 定位：掌握用SQL评估运营活动效果的核心方法论，学会区分增量、时间转移和自然增长
> 前置依赖：W5窗口函数/W6D1复杂SQL/W6D2用户画像 | 后续用途：W7-W8 AB实验的对照组设计
> 面试地位：运营分析面试必考题，"活动效果怎么评估"是经典问题

---

## 📘 区块1：核心概念与第一性原理

### 1.1 活动评估的核心困境

运营做了一个"满200减30"的活动。活动期间GMV涨了40%。老板问："这40%里有多少是活动带来的真增量？"

这个问题看似简单，实则是一个因果推断问题。GMV增长可能来自三个来源：
1. **真实增量**：本来不打算买的人，因为活动才购买
2. **时间转移**：本来就要买的人，提前或推迟到活动期间购买
3. **自然增长**：即使没有活动，GMV也会增长(比如周末效应、月初发工资)

SQL不能直接做因果推断(那需要AB实验和随机化)，但可以通过活动前/中/后的对比，以及活动用户vs非活动用户的对比，提供关键线索。一个好的活动评估SQL分析，至少能排除掉"明显不是活动效果"的增长。

### 1.2 活动评估的三个黄金对比维度

**维度1：时间对比(活动前 vs 活动中 vs 活动后)**
同一个人群，在不同时间段的表现对比。控制用户差异，揭示时间效应。
关键问题：活动后GMV是否暴跌？如果暴跌，说明只是"提前消费"而非增量。

**维度2：人群对比(活动用户 vs 非活动用户)**
同一时间段，不同人群的表现对比。控制时间效应，揭示活动效应。
关键问题：活动用户的消费是否显著高于非活动用户？如果是，活动吸引了消费。

**维度3：交叉对比(时间 × 人群)——DID方法**
DID(Difference-in-Differences)：活动用户的活动前后变化 - 非活动用户的活动前后变化 = 活动净效应。
这是最接近因果推断的观察性方法——同时排除时间效应和人群差异。

### 1.3 活动效果评估的指标体系

| 指标类别 | 核心指标 | 计算方式 | 评估目的 |
|---------|---------|---------|--------|
| 总量指标 | GMV、订单数、用户数 | SUM/COUNT | 看整体规模变化 |
| 人均指标 | 客单价、人均订单数 | AVG | 看用户质量变化 |
| 结构指标 | 新老用户占比、品类分布 | GROUP BY | 看构成是否健康 |
| 效率指标 | ROI(增量GMV/活动成本) | 增量计算 | 看投放效率 |
| 质量指标 | 活动用户后续留存率 | 留存率 | 看长期价值 |

### 1.4 DID(Difference-in-Differences)的基本逻辑

DID的灵感来自自然科学中的"对照实验"思维：
```
              活动前    活动中    变化
活动用户组      A1       A2      A2-A1 = 活动用户变化(含活动效果+自然增长)
非活动用户组    B1       B2      B2-B1 = 自然变化(仅有自然增长)
净效应(DID) = (A2-A1) - (B2-B1) = 排除了自然增长的纯活动效果
```

DID的关键假设是"平行趋势假设"：如果没有活动，活动组和对照组的变化趋势应该相同。这个假设在现实中经常不成立——参与活动的用户本身就更活跃——这就是为什么真正的AB实验需要随机分流(W7-W8会详细学)。

### 1.5 面试高频考点

**面试官常问**：
- "活动期间GMV涨了40%，怎么判断是活动效果还是自然增长？"
- "什么是DID？怎么用SQL实现？"
- "如何识别用户是提前消费还是增量消费？"
- "活动ROI怎么算？成本包括哪些？"
- "如果活动没有对照组怎么评估？"

**回答要点**：
- 不能只看活动期间的绝对值，要对比活动前/中/后三个时段
- DID = (活动组变化 - 对照组变化)，排除自然增长
- 看活动后GMV是否暴跌：暴跌=提前消费，稳健=真增量
- ROI = 增量GMV × 毛利率 / 活动总成本(含券成本+人力+物料+平台费用)
- 无对照组时，用"活动前同时段"作为自身的对照组(前后对比)

## 🧩 区块2：知识点详细拆解

### 2.1 活动前/中/后三段对比法

**为什么重要**：这是活动评估最基础也最常用的方法。三段对比能快速判断活动是否有"透支未来"的效应。

**SQL模板**：
```sql
WITH periods AS (
  SELECT user_id,
    SUM(CASE WHEN dt < '2026-01-15' THEN amount ELSE 0 END) AS before_amt,
    SUM(CASE WHEN dt BETWEEN '2026-01-15' AND '2026-01-22' THEN amount ELSE 0 END) AS during_amt,
    SUM(CASE WHEN dt > '2026-01-22' THEN amount ELSE 0 END) AS after_amt,
    COUNT(CASE WHEN dt < '2026-01-15' THEN 1 END) AS before_orders,
    COUNT(CASE WHEN dt BETWEEN '2026-01-15' AND '2026-01-22' THEN 1 END) AS during_orders,
    COUNT(CASE WHEN dt > '2026-01-22' THEN 1 END) AS after_orders
  FROM orders GROUP BY user_id
)
SELECT
  COUNT(*) AS total_users,
  ROUND(AVG(before_amt), 1) AS avg_before,
  ROUND(AVG(during_amt), 1) AS avg_during,
  ROUND(AVG(after_amt), 1) AS avg_after,
  ROUND(AVG(during_amt)/NULLIF(AVG(before_amt),0)-1, 3) AS during_vs_before,
  ROUND(AVG(after_amt)/NULLIF(AVG(during_amt),0)-1, 3) AS after_vs_during
FROM periods;
```

**结果解读指南**：
- during_vs_before > 0 且 after_vs_during ≈ 0：活动带来增量，且没有透支——理想情况
- during_vs_before > 0 但 after_vs_during < -0.2：活动有效但透支了未来消费——活动后暴跌了20%以上
- during_vs_before ≈ 0：活动基本无效——可能门槛太高或宣传不够
- during_vs_before > 0 且 after_vs_during > 0：活动不仅有效且激活了用户长期消费——最佳情况

### 2.2 活动用户 vs 非活动用户对比

**为什么重要**：排除时间效应(比如活动恰好在周末或发工资日)。

**SQL模板**：
```sql
WITH user_groups AS (
  SELECT user_id,
    MAX(CASE WHEN used_coupon = 1 THEN 1 ELSE 0 END) AS is_promo_user
  FROM orders
  WHERE order_date BETWEEN '2026-01-15' AND '2026-01-22'
  GROUP BY user_id
),
user_spend AS (
  SELECT o.user_id,
    SUM(CASE WHEN o.order_date BETWEEN '2026-01-15' AND '2026-01-22'
             THEN o.amount ELSE 0 END) AS during_spend,
    SUM(o.amount) AS total_spend,
    COUNT(*) AS total_orders
  FROM orders o GROUP BY o.user_id
)
SELECT ug.is_promo_user,
  COUNT(DISTINCT ug.user_id) AS users,
  ROUND(AVG(us.during_spend), 1) AS avg_during_spend,
  ROUND(AVG(us.total_spend), 1) AS avg_total_spend,
  ROUND(AVG(us.total_orders), 1) AS avg_orders
FROM user_groups ug
JOIN user_spend us ON ug.user_id = us.user_id
GROUP BY ug.is_promo_user;
```

**分析的局限性**：活动用户本身就更活跃(选择偏差)，所以不能简单比较绝对值。应该结合DID方法或AB实验来消除选择偏差。

### 2.3 DID(Difference-in-Differences)的完整SQL实现

**SQL实现**：
```sql
WITH did_data AS (
  SELECT o.user_id,
    MAX(CASE WHEN p.used_coupon = 1 THEN 1 ELSE 0 END) AS is_treatment,
    SUM(CASE WHEN o.order_date < '2026-01-15' THEN o.amount ELSE 0 END) AS pre_spend,
    SUM(CASE WHEN o.order_date BETWEEN '2026-01-15' AND '2026-01-22'
             THEN o.amount ELSE 0 END) AS post_spend
  FROM orders o
  LEFT JOIN promo_usage p ON o.user_id = p.user_id
  WHERE o.order_date < '2026-01-23'
  GROUP BY o.user_id
),
group_stats AS (
  SELECT is_treatment,
    ROUND(AVG(pre_spend), 2) AS avg_pre,
    ROUND(AVG(post_spend), 2) AS avg_post,
    ROUND(AVG(post_spend) - AVG(pre_spend), 2) AS change,
    COUNT(*) AS n
  FROM did_data
  WHERE pre_spend > 0 OR post_spend > 0  -- 至少有活动前或活动中的消费
  GROUP BY is_treatment
)
SELECT
  MAX(CASE WHEN is_treatment=1 THEN change END) AS treatment_change,
  MAX(CASE WHEN is_treatment=0 THEN change END) AS control_change,
  MAX(CASE WHEN is_treatment=1 THEN change END)
  - MAX(CASE WHEN is_treatment=0 THEN change END) AS did_effect
FROM group_stats;
```

**DID结果解读**：
- did_effect > 0：活动可能带来了正增量(但需要注意统计显著性和平行趋势假设)
- did_effect ≈ 0：活动效果不明显
- did_effect < 0：活动可能有负面影响(比如抬高门槛反而抑制了消费)

**DID的三大假设**：
1. 平行趋势假设：无活动时两组变化趋势相同(无法完全验证但可以看活动前趋势)
2. 无溢出效应：活动不影响对照组(社交电商中这个假设经常被违反)
3. 稳定单元处理值(SUTVA)：每个用户只受自己是否参与的影响

### 2.4 薅羊毛用户识别

**为什么重要**：如果活动用户中有大量只买刚好满门槛金额的订单，说明用户主要在薅羊毛而非正常消费。这些用户的GMV是"虚"的——活动一停就走了。

**识别逻辑**：
- 活动门槛200元 → 观察订单金额在200-210元之间的占比
- 如果大量订单集中在门槛附近(200-210元占30%+)，这是明显的薅羊毛信号
- 如果订单金额分布和平时分布相似(呈对数正态分布)，说明用户在正常消费

```sql
SELECT
  CASE WHEN amount BETWEEN 200 AND 210 THEN '门槛附近(200-210)'
       WHEN amount BETWEEN 211 AND 300 THEN '略超门槛(211-300)'
       WHEN amount > 300 THEN '远超门槛(300+)'
       ELSE '未达门槛(<200)' END AS order_range,
  COUNT(*) AS order_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM orders
WHERE order_date BETWEEN '2026-01-15' AND '2026-01-22'
GROUP BY order_range
ORDER BY MIN(amount);
```

**薅羊毛用户的其他特征**：
- 订单金额高度集中在门槛附近(变异系数很小)
- 活动前很少购买，活动中突然出现，活动后立即消失
- 使用新注册账号或小号
- 购买的商品多为标品(容易转卖)

### 2.5 活动ROI计算

**为什么重要**：老板最终关心的是花了多少钱、赚了多少钱。ROI是决定活动是否"值得继续做"的关键指标。

**ROI公式详解**：
```
活动ROI = (增量GMV × 毛利率 - 活动总成本) / 活动总成本 × 100%

其中：
- 增量GMV = DID效应 × 活动参与用户数(只有增量部分)
- 毛利率 = (GMV - 商品成本) / GMV(通常15-50%，取决于品类)
- 活动总成本 = 优惠券使用金额 + 运营人力成本 + 宣传物料成本 + 平台技术费用
```

**ROI > 0%**：活动盈利，值得继续
**ROI = 0%**：活动打平，要考虑品牌曝光等隐性收益
**ROI < 0%**：活动亏损，需要优化或停止(除非是战略性拉新)

### 2.6 活动后留存追踪

**为什么重要**：活动可能带来一次性的GMV暴涨，但如果活动用户很快就流失了，长期ROI可能是负的。

**SQL实现(7日/30日留存)**：
```sql
WITH promo_users AS (
  SELECT DISTINCT user_id FROM orders
  WHERE order_date BETWEEN '2026-01-15' AND '2026-01-22'
    AND is_promo_order = 1
)
SELECT
  COUNT(DISTINCT p.user_id) AS promo_users,
  COUNT(DISTINCT o.user_id) AS retained_7d,
  ROUND(COUNT(DISTINCT o.user_id) * 100.0
    / COUNT(DISTINCT p.user_id), 1) AS retention_7d_pct,
  COUNT(DISTINCT o2.user_id) AS retained_30d,
  ROUND(COUNT(DISTINCT o2.user_id) * 100.0
    / COUNT(DISTINCT p.user_id), 1) AS retention_30d_pct
FROM promo_users p
LEFT JOIN orders o ON p.user_id = o.user_id
  AND o.order_date BETWEEN '2026-01-23' AND '2026-01-29'
LEFT JOIN orders o2 ON p.user_id = o2.user_id
  AND o2.order_date BETWEEN '2026-01-23' AND '2026-02-21';
```

### 2.7 活动评估报告的完整结构

一个专业的活动评估报告应该包含以下七个部分：
1. **活动概览**：时间、人群范围、活动策略(满减/折扣/秒杀)、预算和实际成本
2. **总量效果**：GMV、订单数、用户数、客单价的变化(活动期 vs 对比期)
3. **增量分析**：DID净效应、自然增长剥离、ROI计算
4. **结构分析**：新老用户贡献变化、品类分布变化、渠道贡献变化
5. **质量分析**：活动用户的7日/30日留存、LTV变化、复购率
6. **异常识别**：薅羊毛用户占比、刷单风险、数据异常波动
7. **结论与建议**：活动是否值得继续/扩大/优化/停止，具体改进建议

### 2.8 活动评估中的常见分析误区

**误区1：只看活动期间的绝对值**
"活动期间GMV 100万"——但没有对比就没有结论。对比期选错了(比如选了淡季)，结论可能完全相反。

**误区2：混淆相关性(相关)和因果性(导致)**
"活动期间，参与活动的用户平均消费500元，未参与的用户平均消费200元"——这不说明活动效果好，因为参与活动的用户本身就更活跃(选择偏差)。

**误区3：忽略活动后的回落**
"活动期间GMV涨了50%"——但如果活动后GMV暴跌60%，总GMV可能还是亏的。

**误区4：忽略隐性成本**
只算优惠券成本，忘了运营策划花了3个人2周的时间——人力成本往往比券成本更高。

**误区5：用ROI代替长期价值**
拉新活动的短期ROI可能是负的，但如果新用户的长期LTV高，整体还是值得的。评估周期要和业务目标匹配。

## 💻 区块3：动手练习

### 步骤0：数据准备
模拟活动数据：1月1-14活动前，1月15-22活动中(消费上浮30%)，1月23-31活动后
创建活动参与标记来模拟分析场景

In [ ]:
import sqlite3, pandas as pd, numpy as np
np.random.seed(42); conn=sqlite3.connect(':memory:')

# 模拟活动数据：1月1-14 活动前，1月15-22 活动中，1月23-31 活动后
n=800; users=range(1,101)
data=[]
for uid in users:
    base=np.random.lognormal(3.5,0.5)
    for _ in range(np.random.poisson(8)):
        d=np.random.choice(pd.date_range('2026-01-01','2026-01-31'))
        amt=np.random.lognormal(4,0.7)
        # 活动期间消费略高
        if '2026-01-15'<=d.strftime('%Y-%m-%d')<='2026-01-22':
            amt*=1.3
        data.append((uid,d.strftime('%Y-%m-%d'),round(amt,2)))

pd.DataFrame(data,columns=['user_id','order_date','amount']).to_sql('orders',conn,index=False)

print("活动前(1-14) vs 活动中(15-22) vs 活动后(23-31)")
q="""SELECT
  CASE WHEN order_date<'2026-01-15' THEN '活动前'
       WHEN order_date<='2026-01-22' THEN '活动中'
       ELSE '活动后' END AS period,
  COUNT(DISTINCT user_id) AS users,
  ROUND(SUM(amount),0) AS total_gmv,
  ROUND(AVG(amount),1) AS avg_order
FROM orders GROUP BY period"""
print(pd.read_sql(q,conn))

print("\n关键观察：活动期间GMV确实涨了，但活动后是否回落？")
print("如果活动后GMV暴跌→说明只是'提前消费'，不是真增量")
print("如果活动后GMV维持→说明活动带来了新用户/激活了沉睡用户")


### 步骤2：进阶练习——用户级别的活动前后对比

2.1 每个用户在三个时期的人均消费变化
2.2 识别"只在活动期间购买"的一次性用户(薅羊毛嫌疑)
2.3 用户消费变化分布分析

In [ ]:
# 步骤2.1：每个用户三个时段的消费
q_user_periods = """SELECT user_id,
  SUM(CASE WHEN order_date<'2026-01-15' THEN amount ELSE 0 END) AS before_amt,
  SUM(CASE WHEN order_date BETWEEN '2026-01-15' AND '2026-01-22' THEN amount ELSE 0 END) AS during_amt,
  SUM(CASE WHEN order_date>'2026-01-22' THEN amount ELSE 0 END) AS after_amt
FROM orders GROUP BY user_id"""
user_periods = pd.read_sql(q_user_periods, conn)
print("用户三时段消费(前10)：")
print(user_periods.head(10).to_string())
print(f"\n活动前有消费: {(user_periods['before_amt']>0).sum()}人")
print(f"活动中有消费: {(user_periods['during_amt']>0).sum()}人")
print(f"活动后有消费: {(user_periods['after_amt']>0).sum()}人")

# 步骤2.2：识别一次性用户
print("\n只在活动期间购买的用户(潜在薅羊毛)：")
q_onetime = """SELECT COUNT(*) AS onetime_users FROM (
  SELECT user_id,
    SUM(CASE WHEN order_date<'2026-01-15' THEN 1 ELSE 0 END) AS before_cnt,
    SUM(CASE WHEN order_date BETWEEN '2026-01-15' AND '2026-01-22' THEN 1 ELSE 0 END) AS during_cnt,
    SUM(CASE WHEN order_date>'2026-01-22' THEN 1 ELSE 0 END) AS after_cnt
  FROM orders GROUP BY user_id
) WHERE before_cnt=0 AND during_cnt>0 AND after_cnt=0"""
print(pd.read_sql(q_onetime, conn))

# 步骤2.3：用户消费变化分布
print("\n用户活动前后消费变化分析：")
q_change = """SELECT
  CASE WHEN during_amt > before_amt * 1.5 THEN '大幅增长(>50%)'
       WHEN during_amt > before_amt THEN '小幅增长'
       WHEN during_amt = before_amt THEN '持平'
       WHEN during_amt > 0 THEN '下降'
       ELSE '无活动消费' END AS change_type,
  COUNT(*) AS user_count
FROM (
  SELECT user_id,
    SUM(CASE WHEN order_date<'2026-01-15' THEN amount ELSE 0 END) AS before_amt,
    SUM(CASE WHEN order_date BETWEEN '2026-01-15' AND '2026-01-22' THEN amount ELSE 0 END) AS during_amt
  FROM orders GROUP BY user_id
) WHERE before_amt > 0
GROUP BY change_type ORDER BY user_count DESC"""
print(pd.read_sql(q_change, conn))


### 步骤3：挑战练习——简易DID效果估算

模拟活动参与标记(消费较高的用户更可能参与活动)，用DID方法估算活动的净增量效应

In [ ]:
# 步骤3：简易DID效果估算
# 首先创建活动参与标记(模拟：消费较高的用户更可能参与活动)
conn.execute("""CREATE TABLE promo_flag AS
SELECT user_id,
  CASE WHEN SUM(amount) > (SELECT AVG(total) FROM (
    SELECT SUM(amount) AS total FROM orders GROUP BY user_id
  )) THEN 1 ELSE 0 END AS is_promo
FROM orders GROUP BY user_id""")

print("DID效果估算：")
q_did = """WITH did_data AS (
  SELECT o.user_id, MAX(p.is_promo) AS is_treatment,
    SUM(CASE WHEN o.order_date<'2026-01-15' THEN o.amount ELSE 0 END) AS pre_spend,
    SUM(CASE WHEN o.order_date BETWEEN '2026-01-15' AND '2026-01-22'
             THEN o.amount ELSE 0 END) AS post_spend
  FROM orders o
  LEFT JOIN promo_flag p ON o.user_id = p.user_id
  GROUP BY o.user_id
),
group_stats AS (
  SELECT is_treatment,
    AVG(pre_spend) AS avg_pre,
    AVG(post_spend) AS avg_post,
    AVG(post_spend)-AVG(pre_spend) AS change,
    COUNT(*) AS n
  FROM did_data GROUP BY is_treatment
)
SELECT is_treatment,
  ROUND(avg_pre,0) AS avg_pre,
  ROUND(avg_post,0) AS avg_post,
  ROUND(change,0) AS avg_change,
  n AS user_count
FROM group_stats"""
did_result = pd.read_sql(q_did, conn)
print(did_result.to_string())
treatment_change = did_result[did_result['is_treatment']==1]['avg_change'].values[0]
control_change = did_result[did_result['is_treatment']==0]['avg_change'].values[0]
print(f"\nDID净效应 = {treatment_change:.0f} - ({control_change:.0f}) = {treatment_change - control_change:.0f}")
print("这个数值代表排除了自然增长后的'活动净增量'")
print("\n注意：这里对照组的变化可能非零(自然增长)，DID排除了这部分的干扰")


## 📊 区块4：业务场景应用

### 场景1：满减活动效果评估

**业务需求**：运营做了"满200减30"活动(1月15-22日)，需要评估活动效果。

**完整分析步骤**：
1. 总量对比：活动期间(15-22日) vs 活动前同时长时段(7-14日)
2. 结构对比：新老用户贡献变化、客单价分布变化、品类分布变化
3. 薅羊毛检测：订单金额在200-210元的占比是否异常
4. 活动后追踪：活动后7天GMV是否回落(判断透支效应)
5. ROI计算：增量GMV(用DID) vs 优惠券总成本

**关键SQL**：
```sql
WITH analysis AS (
  SELECT '活动前' AS period,
    COUNT(*) AS orders, SUM(amount) AS gmv,
    AVG(amount) AS avg_order,
    COUNT(DISTINCT user_id) AS users
  FROM orders WHERE order_date BETWEEN '2026-01-07' AND '2026-01-14'
  UNION ALL
  SELECT '活动中' AS period,
    COUNT(*) AS orders, SUM(amount) AS gmv,
    AVG(amount) AS avg_order,
    COUNT(DISTINCT user_id) AS users
  FROM orders WHERE order_date BETWEEN '2026-01-15' AND '2026-01-22'
)
SELECT * FROM analysis;
```

### 场景2：签到活动效果分析

**业务需求**：连续签到7天领红包，评估签到活动是否提升了用户活跃度。

**分析方法**：
- 对比签到用户和未签到用户在活动后7天/30天的留存率
- 如果签到用户的后续留存显著更高，说明签到筛选出了高粘性用户
- 但不能说签到"造成"了高留存(选择偏差——本来就更活跃的用户才参加签到)
- 要验证因果效果，需要AB实验：随机分配用户是否能参加签到活动

### 场景3：拉新活动的渠道效果对比

**业务需求**：在不同渠道投放了"新人首单5折"活动，需要对比各渠道的获客ROI。

**SQL实现**：
```sql
WITH channel_new_users AS (
  SELECT u.channel, u.user_id,
    MIN(o.order_date) AS first_order_date,
    SUM(o.amount) AS first_month_spend
  FROM users u
  JOIN orders o ON u.user_id = o.user_id
  WHERE u.register_date >= '2026-01-15'
    AND o.order_date <= DATE_ADD(u.register_date, INTERVAL 30 DAY)
  GROUP BY u.channel, u.user_id
)
SELECT channel,
  COUNT(*) AS new_users,
  ROUND(AVG(first_month_spend), 0) AS avg_first_month_spend,
  ROUND(AVG(DATEDIFF(first_order_date, '2026-01-15')), 1) AS avg_days_to_first_order
FROM channel_new_users
GROUP BY channel
ORDER BY avg_first_month_spend DESC;
```

### 场景4：活动对用户长期价值的影响

**业务需求**：评估某次活动带来的用户，30天后的留存和LTV是否优于自然用户。

**分析方法(同期群对比)**：
1. 找出活动期间首次购买的用户(活动新客)
2. 找出活动前30天首次购买的用户(自然新客，作为对照组)
3. 对比两组用户注册后30天/60天/90天内的累计消费
4. 如果活动新客的LTV显著低于自然新客 → 活动吸引的是"羊毛党"
5. 如果活动新客的LTV不低于自然新客 → 活动带来了健康的增量用户

**核心洞察**：活动的短期GMV增长可能掩盖长期的用户质量问题。好的活动评估不仅要看"活动期间发生了什么"，更要看"活动结束后用户去了哪里"。

## ✅ 区块5：分级检验清单

### 🟢 初级检验(必须能独立完成的)
- [ ] 能写出活动前/中/后三段对比的SQL
- [ ] 能计算活动期间的GMV变化率(活动中 vs 活动前)
- [ ] 能区分增量、时间转移、自然增长三个概念
- [ ] 能识别"只在活动期间购买"的一次性用户
- [ ] 能说出活动评估的五个核心指标类别

**答案提示**：
- 三段对比：SUM(CASE WHEN period判断 THEN amount ELSE 0 END)
- 变化率：(活动中 - 活动前) / 活动前 × 100%
- 增量=离开活动不会发生的消费；时间转移=早晚要买只是换了时间
- 一次性用户：before_cnt=0 AND during_cnt>0 AND after_cnt=0
- 五类指标：总量、人均、结构、效率(ROI)、质量(留存)

### 🟡 中级检验(能解释给同事听的原理)
- [ ] 能解释DID的基本逻辑和SQL实现
- [ ] 能识别薅羊毛用户(订单金额集中在门槛附近)
- [ ] 能计算活动的简单ROI(包含毛利率和全部成本)
- [ ] 能写出活动用户vs非活动用户的对比SQL
- [ ] 能分析活动后GMV的变化趋势及含义

**答案提示**：
- DID = (活动组变化) - (对照组变化)，排除自然增长和选择偏差
- 薅羊毛：订单金额在门槛附近(±5%)的占比超过30%
- ROI = (增量GMV × 毛利率 - 总成本) / 总成本
- 活动后暴跌=提前消费，稳健/继续增长=真增量+激活效应

### 🔴 高级检验(能举一反三的迁移能力)
- [ ] 能设计完整的活动评估报告模板(7个部分)
- [ ] 能结合画像标签做活动效果的分群分析
- [ ] 能评估活动的长期LTV影响(而非只看短期GMV)
- [ ] 能对多期活动做横向对比分析(哪种活动ROI最高)
- [ ] 能识别活动评估中的5个常见误区并提出改进方案

**答案提示**：
- 完整报告：活动概览、总量效果、增量分析、结构分析、质量分析、异常识别、结论建议
- 分群分析：按用户价值分层看活动效果差异
- LTV评估：追踪3/6/12个月的消费轨迹
- 多期对比：统一指标口径，横向对比不同类型活动的ROI
- 5个误区：只看绝对值、混淆相关与因果、忽略活动后回落、忽略隐性成本、用ROI代替长期价值

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W6D2 用户画像 | 活动效果需要按画像分群对比 |
| **← 前置依赖** | W5D2 窗口函数LAG | 活动前后对比需要LAG计算环比 |
| **← 前置依赖** | W6D1 复杂SQL拆解 | 活动评估是多CTE链的典型应用 |
| **→ 后续用途** | W7-W8 AB实验 | 活动评估是AB实验的简化版(无随机化) |
| **→ 后续用途** | W9 因果推断 | DID是因果推断的入门方法 |
| **业务连接** | 运营活动ROI评估 | 直接用于活动预算决策 |
| **业务连接** | 用户增长策略 | 评估拉新/留存活动的效率 |

### 横向连接

**与AB实验的连接**：活动评估是"事后分析"(没有随机化)，AB实验是"事前设计"(随机分流)。AB实验通过随机化消除选择偏差，得到更可信的因果结论。W7-W8会详细学习。

**与因果推断的连接**：DID是因果推断的入门方法。W9会学习更多方法：匹配法(Propensity Score Matching)、断点回归(Regression Discontinuity)、工具变量(Instrumental Variables)。

**与财务分析的连接**：活动ROI计算直接关联财务指标。需要和财务团队对齐毛利率口径和成本归因方式。跨部门协作是高级数据分析师的必备能力。

**与产品分析的连接**：活动效果往往反映产品本身的问题。如果"不管做什么活动，用户都不来"——问题可能不在活动，而在产品本身缺乏价值。

## 📝 区块7：当日小结

> **活动效果评估要区分"自然增长"、"时间转移"和"真实增量"。三段对比法(活动前/中/后)是基础，DID(双重差分)是进阶——用活动组的变化减去对照组的变化。SQL不能做因果推断(需要随机化)，但可以用数据对比提供关键线索。ROI计算要包含增量GMV、毛利率和全部成本。活动后留存追踪决定用户的长期价值——短期GMV暴涨+活动后暴跌 = 透支未来，不是真增量。**

**三条金句带回家**：
1. "活动期间GMV涨了"不能证明活动有效——要先排除自然增长和时间转移
2. DID = (活动组变化 - 对照组变化)，这是最接近科学的活动评估方法
3. 薅羊毛用户的特征是订单金额集中在活动门槛附近——用订单金额分布来识别

### 🔖 课后拓展

1. **练习**：设计一个完整的活动评估SQL，包含三段对比、DID效果估算、用户分群分析和ROI计算

2. **思考**：如果活动没有对照组(所有用户都参与了活动)，怎么评估活动效果？(提示：用活动前数据作为自身的"对照组"，或找同期未做活动的相似产品作为"外部对照组")

3. **预习**：明天的经典分析题训练，将用到今天学的对比分析思维来解决面试中的综合题

4. **面试准备**：准备回答"如何评估一个运营活动的效果？"——从数据收集→指标设计→三段对比→DID分析→ROI计算→报告输出，完整的分析框架

---

*03analyst_thinking · Week 6 Day 3 · 2026-05-22*